# Cognitive Intervention & Working Memory Capacity Analysis

**Study Design:** Pre-test / Post-test with Experimental vs Control groups  
**Cognitive Tests:** Corsi Block-Tapping (Visuospatial), Digit Span Forward (Verbal), Ospan (Complex WM)  
**Academic Outcome:** Content knowledge test (Pre & Post)  

---

## Pipeline Overview
| Phase | Description |
|-------|-------------|
| **1** | Data Integration & Preprocessing |
| **2** | Feature Engineering (Z-scores, WMC Composite, WMC Level, Learning Gain) |
| **3** | Statistical Analysis (Two-Way Between-Subjects ANOVA -- H1, Moderated Regression -- H2) |
| **4** | Machine Learning (Random Forest, SVM, XGBoost with 10-Fold CV) |

> **Public-repository note:** Notebook outputs have been cleared and direct personal identifiers have been excluded from the public analysis workflow. Raw participant-level data are not included in this repository.


In [ ]:

# !pip install pandas numpy scipy scikit-learn statsmodels pingouin xgboost matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Stats
from scipy import stats
from scipy.stats import shapiro, levene
import pingouin as pg
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

# ML
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score
)
from sklearn.model_selection import cross_val_predict

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.1)
 

---
## Phase 1 — Data Integration & Preprocessing

In [ ]:
# -- 1.1  Load all five CSV files ---
DATA_DIR = 'data/'

corsi   = pd.read_csv(DATA_DIR + 'Corsi data.csv')
digit   = pd.read_csv(DATA_DIR + 'Digit Span data.csv')
ospan   = pd.read_csv(DATA_DIR + 'Ospan_data.csv')
pretest = pd.read_csv(DATA_DIR + 'Pre and Post-Test data-PRE-TEST.csv')
posttest = pd.read_csv(DATA_DIR + 'Pre and Post-Test data-POST TEST.csv')

print(f'Corsi:    {corsi.shape}')
print(f'Digit:    {digit.shape}')
print(f'Ospan:    {ospan.shape}')
print(f'Pre-Test: {pretest.shape}')
print(f'Post-Test:{posttest.shape}')

In [ ]:
# -- 1.2  Standardise column names & select key columns ---

# Shared demographic columns (taken from one source later)
# Direct identifiers (e.g., student name and national ID) are intentionally excluded.
DEMO_COLS = ['# ms_ID', 'grade_name', 'class_name', 'nationality_name',
             'gender_id', 'Group', 'age_1']

# Corsi
corsi = corsi.rename(columns={'Corsi span_score': 'Corsi_Score'})
corsi_slim = corsi[DEMO_COLS + ['Corsi_Score']].copy()

# Digit Span
digit = digit.rename(columns={'Digit Span_Score': 'DigitSpan_Score'})
digit_slim = digit[['# ms_ID', 'DigitSpan_Score']].copy()

# Ospan (keep math accuracy for QC filter)
ospan_slim = ospan[['# ms_ID', 'Ospan_Score', 'Correct math proplems%']].copy()

# Pre-Test
pretest = pretest.rename(columns={'score (100)': 'Pre_Content_Score'})
pretest_slim = pretest[['# ms_ID', 'Pre_Content_Score']].copy()

# Post-Test
posttest = posttest.rename(columns={'score (100)': 'Post_Content_Score'})
posttest_slim = posttest[['# ms_ID', 'Post_Content_Score']].copy()

print('Column renaming done')
print('  Corsi key column : Corsi_Score')
print('  Digit key column : DigitSpan_Score')
print('  Ospan key columns: Ospan_Score, Correct math proplems%')
print('  Pre-Test column  : Pre_Content_Score')
print('  Post-Test column : Post_Content_Score')

In [ ]:
# -- 1.3  Merge all datasets on student ID (inner join) ---
df = (
    corsi_slim
    .merge(digit_slim,    on='# ms_ID', how='inner')
    .merge(ospan_slim,    on='# ms_ID', how='inner')
    .merge(pretest_slim,  on='# ms_ID', how='inner')
    .merge(posttest_slim, on='# ms_ID', how='inner')
)

print(f'Merged dataset shape: {df.shape}')
print(f'Students retained after inner join: {len(df)}')

In [ ]:
# -- 1.4  Quality-Control: Filter Ospan math accuracy < 85% ---
n_before_qc = len(df)
df['Correct math proplems%'] = pd.to_numeric(df['Correct math proplems%'], errors='coerce')

low_math = df[df['Correct math proplems%'] < 85]
print(f'Students with Ospan math accuracy < 85%: {len(low_math)}')
# Do not print row-level identifiers in the public reproducibility notebook.

df = df[df['Correct math proplems%'] >= 85].copy()
n_excluded_qc = n_before_qc - len(df)
print(f'\nStudents removed: {n_excluded_qc}')
print(f'Students remaining: {len(df)}')

In [ ]:
# -- 1.5  Data type conversions & missing-value check ---
numeric_cols = ['Corsi_Score', 'DigitSpan_Score', 'Ospan_Score',
                'Pre_Content_Score', 'Post_Content_Score', 'age_1']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal rows with any NaN: {df.isnull().any(axis=1).sum()}')

# Drop rows with NaN in key analysis columns (if any)
n_before = len(df)
df = df.dropna(subset=numeric_cols).copy()
print(f'Rows dropped due to NaN: {n_before - len(df)}')
print(f'Final clean dataset: {len(df)} students')

In [ ]:
# -- 1.6  Group distribution & descriptive statistics ---
print('Group counts')
print(df['Group'].value_counts())
print()

print('Gender counts')
print(df['gender_id'].value_counts())
print()

print('Grade distribution')
print(df['grade_name'].value_counts())
print()

print('Descriptive Statistics (numeric columns)')
df[numeric_cols].describe().round(2)

---
## Phase 2 — Feature Engineering

In [ ]:
# -- 2.1  Z-scores for each cognitive measure ---
df['z_Corsi']     = stats.zscore(df['Corsi_Score'])
df['z_DigitSpan'] = stats.zscore(df['DigitSpan_Score'])
df['z_Ospan']     = stats.zscore(df['Ospan_Score'])

print('Z-score descriptives:')
df[['z_Corsi', 'z_DigitSpan', 'z_Ospan']].describe().round(3)

In [ ]:
# -- 2.2  WMC Composite = mean of three z-scores ---
df['WMC_Composite'] = df[['z_Corsi', 'z_DigitSpan', 'z_Ospan']].mean(axis=1)

print(f'WMC_Composite  mean={df["WMC_Composite"].mean():.4f}  '
      f'std={df["WMC_Composite"].std():.4f}  '
      f'min={df["WMC_Composite"].min():.4f}  '
      f'max={df["WMC_Composite"].max():.4f}')

In [ ]:
# -- 2.3  Correlation Matrix: Cognitive Measures ---
cog_cols = ['Corsi_Score', 'DigitSpan_Score', 'Ospan_Score', 'WMC_Composite']
corr_matrix = df[cog_cols].corr()

print('Pearson Correlation Matrix')
print(corr_matrix.round(3))

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
            vmin=-1, vmax=1, center=0, mask=mask,
            square=True, linewidths=0.5, ax=ax,
            xticklabels=['Corsi', 'Digit Span', 'Ospan', 'WMC Composite'],
            yticklabels=['Corsi', 'Digit Span', 'Ospan', 'WMC Composite'])
ax.set_title('Cognitive Measures - Correlation Matrix')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: correlation_heatmap.png')

In [ ]:
# -- 2.4  WMC Level (tertile split: Low / Medium / High) ---
df['WMC_Level'] = pd.qcut(
    df['WMC_Composite'], q=3, labels=['Low', 'Medium', 'High']
)

print('WMC Level distribution:')
print(df['WMC_Level'].value_counts().sort_index())
print()
print('WMC Level x Group crosstab:')
pd.crosstab(df['WMC_Level'], df['Group'])

In [ ]:
# -- 2.5  Learning Gain = Post - Pre ---
df['Learning_Gain'] = df['Post_Content_Score'] - df['Pre_Content_Score']

print(f'Learning Gain  mean={df["Learning_Gain"].mean():.2f}  '
      f'std={df["Learning_Gain"].std():.2f}  '
      f'min={df["Learning_Gain"].min():.2f}  '
      f'max={df["Learning_Gain"].max():.2f}')
print()
print('Learning Gain by Group:')
df.groupby('Group')['Learning_Gain'].describe().round(2)

In [ ]:
# -- 2.5  Outlier Detection (IQR method) ---
outlier_vars = ['Corsi_Score', 'DigitSpan_Score', 'Ospan_Score',
                'Pre_Content_Score', 'Post_Content_Score', 'Learning_Gain']

outlier_summary = []
for col in outlier_vars:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary.append({
        'Variable': col,
        'Q1': round(Q1, 2),
        'Q3': round(Q3, 2),
        'IQR': round(IQR, 2),
        'Lower fence': round(lower, 2),
        'Upper fence': round(upper, 2),
        'N outliers': n_outliers,
        '% outliers': round(100 * n_outliers / len(df), 1)
    })

outlier_df = pd.DataFrame(outlier_summary)
print('Outlier Detection (1.5 x IQR method)')
print(outlier_df.to_string(index=False))

# Decision: retain outliers (flag only)
# Rationale: sample is large (N=380), outliers reflect genuine variation,
# and removal could bias results. Outliers are documented but kept.
print('\nDecision: Outliers RETAINED in analysis.')
print('Rationale: Genuine data variability in a large sample (N={}).'.format(len(df)))
print('Extreme values are flagged but not removed to avoid selection bias.')

In [ ]:
# -- 2.7  Visualise distributions ---
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Row 1: Cognitive measures
for ax, col, title in zip(
    axes[0],
    ['Corsi_Score', 'DigitSpan_Score', 'Ospan_Score'],
    ['Corsi (Visuospatial)', 'Digit Span Forward (Verbal)', 'Ospan (Complex WM)']
):
    sns.histplot(df, x=col, hue='Group', kde=True, ax=ax, bins=15)
    ax.set_title(title)

# Row 2: Pre/Post scores & Learning Gain
sns.histplot(df, x='Pre_Content_Score', hue='Group', kde=True, ax=axes[1, 0], bins=15)
axes[1, 0].set_title('Pre-Test Score')

sns.histplot(df, x='Post_Content_Score', hue='Group', kde=True, ax=axes[1, 1], bins=15)
axes[1, 1].set_title('Post-Test Score')

sns.histplot(df, x='Learning_Gain', hue='Group', kde=True, ax=axes[1, 2], bins=15)
axes[1, 2].set_title('Learning Gain (Post - Pre)')

plt.tight_layout()
plt.savefig('distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: distributions.png')

In [ ]:
# -- 2.8  Boxplot: Learning Gain by Group x WMC Level ---
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='WMC_Level', y='Learning_Gain', hue='Group',
            order=['Low', 'Medium', 'High'], ax=ax)
ax.set_title('Learning Gain by WMC Level x Group')
ax.set_xlabel('WMC Level (Tertile)')
ax.set_ylabel('Learning Gain')
plt.tight_layout()
plt.savefig('boxplot_gain_by_wmc_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: boxplot_gain_by_wmc_group.png')

---
## Phase 3 — Statistical Analysis

### 3.1  Assumption Checks

In [ ]:
# -- 3.1.1  Shapiro-Wilk normality test per cell (Group x WMC_Level) ---
print(f'{"Group":<15} {"WMC_Level":<10} {"W":>8} {"p-value":>10} {"Normal?":>10}')

normality_results = []
for group in ['Experimental', 'Control']:
    for level in ['Low', 'Medium', 'High']:
        subset = df[(df['Group'] == group) & (df['WMC_Level'] == level)]['Learning_Gain']
        if len(subset) >= 3:
            w, p = shapiro(subset)
            normal = 'Yes' if p > 0.05 else 'No'
            print(f'{group:<15} {level:<10} {w:>8.4f} {p:>10.4f} {normal:>10}')
            normality_results.append({'Group': group, 'WMC_Level': level,
                                       'W': w, 'p': p, 'Normal': normal})
        else:
            print(f'{group:<15} {level:<10}  n={len(subset)} -- too few')

normality_df = pd.DataFrame(normality_results)
all_normal = all(normality_df['Normal'] == 'Yes')
print(f'\nAll cells normally distributed: {all_normal}')

In [ ]:
# -- 3.1.2  Levene's test for homogeneity of variance ---
groups_for_levene = []
for group in ['Experimental', 'Control']:
    for level in ['Low', 'Medium', 'High']:
        subset = df[(df['Group'] == group) & (df['WMC_Level'] == level)]['Learning_Gain']
        if len(subset) >= 2:
            groups_for_levene.append(subset.values)

lev_stat, lev_p = levene(*groups_for_levene)
print(f'Levene\'s test: W={lev_stat:.4f}, p={lev_p:.4f}')
print(f'Homogeneity of variance assumed: {"Yes" if lev_p > 0.05 else "No (violated)"}')

### 3.2  H1 -- Two-Way Between-Subjects ANOVA: Group x WMC_Level -> Learning Gain

Testing whether the **interaction between instructional group and WMC level** significantly affects learning gain.

In [ ]:
# -- 3.2.1  Two-Way ANOVA (Type II) using statsmodels ---
model_anova = ols('Learning_Gain ~ C(Group) * C(WMC_Level)', data=df).fit()
anova_table = anova_lm(model_anova, typ=2)

print('Two-Way ANOVA (Type II)')
print(anova_table.round(4))

In [ ]:
# -- 3.2.2  Effect sizes (Eta-squared & Partial Eta-squared) ---
ss_total = anova_table['sum_sq'].sum()
anova_table['eta_sq'] = anova_table['sum_sq'] / ss_total

# Partial eta-squared = SS_effect / (SS_effect + SS_residual)
ss_resid = anova_table.loc['Residual', 'sum_sq']
anova_table['partial_eta_sq'] = anova_table['sum_sq'] / (anova_table['sum_sq'] + ss_resid)

print('Effect Sizes')
print(anova_table[['sum_sq', 'df', 'F', 'PR(>F)', 'eta_sq', 'partial_eta_sq']].round(4))

In [ ]:
# -- 3.2.3  Post-hoc pairwise comparisons (if interaction is significant) ---
interaction_p = anova_table.loc['C(Group):C(WMC_Level)', 'PR(>F)']

if interaction_p < 0.05:
    print(f'Interaction is SIGNIFICANT (p={interaction_p:.4f}) -- running post-hoc tests\n')

    # Pairwise t-tests within each WMC level (Experimental vs Control)
    for level in ['Low', 'Medium', 'High']:
        exp = df[(df['Group'] == 'Experimental') & (df['WMC_Level'] == level)]['Learning_Gain']
        ctrl = df[(df['Group'] == 'Control') & (df['WMC_Level'] == level)]['Learning_Gain']
        t_stat, p_val = stats.ttest_ind(exp, ctrl)
        d = (exp.mean() - ctrl.mean()) / np.sqrt((exp.std()**2 + ctrl.std()**2) / 2)
        print(f'WMC={level}: Exp(M={exp.mean():.2f}) vs Ctrl(M={ctrl.mean():.2f}) -- '
              f't={t_stat:.3f}, p={p_val:.4f}, Cohen d={d:.3f}')
else:
    print(f'Interaction is NOT significant (p={interaction_p:.4f}) -- no post-hoc needed.')
    print('\nMain effects summary:')
    for effect in ['C(Group)', 'C(WMC_Level)']:
        p = anova_table.loc[effect, 'PR(>F)']
        eta = anova_table.loc[effect, 'partial_eta_sq']
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f'  {effect}: F={anova_table.loc[effect, "F"]:.3f}, '
              f'p={p:.4f} {sig}, partial eta_p={eta:.4f}')

### 3.3  Robustness Check: Welch's ANOVA & Kruskal-Wallis

Since normality (Shapiro-Wilk) and/or homogeneity of variance (Levene's) assumptions were violated,
we run **Welch's ANOVA** (robust to unequal variances) and the non-parametric **Kruskal-Wallis** test
as robustness checks for the Group main effect.

In [ ]:
# -- 3.3.1  Welch's ANOVA (Group -> Learning Gain) ---
welch_result = pg.welch_anova(dv='Learning_Gain', between='Group', data=df)
print('Welch\'s ANOVA: Group -> Learning Gain')
print(welch_result.to_string(index=False))

print()

# -- 3.3.2  Kruskal-Wallis (non-parametric alternative) ---
exp_gains = df[df['Group'] == 'Experimental']['Learning_Gain']
ctrl_gains = df[df['Group'] == 'Control']['Learning_Gain']
kw_stat, kw_p = stats.kruskal(exp_gains, ctrl_gains)
print('Kruskal-Wallis: Group -> Learning Gain')
print(f'H = {kw_stat:.4f}, p = {kw_p:.6f}')

# Mann-Whitney U (pairwise non-parametric)
u_stat, u_p = stats.mannwhitneyu(exp_gains, ctrl_gains, alternative='two-sided')
rank_biserial = 1 - (2 * u_stat) / (len(exp_gains) * len(ctrl_gains))
print(f'\nMann-Whitney U = {u_stat:.1f}, p = {u_p:.6f}, rank-biserial r = {rank_biserial:.4f}')

print('\nRobustness summary:')
print(f'  Standard ANOVA Group p = {anova_table.loc["C(Group)", "PR(>F)"]:.6f}')
print(f'  Welch\'s ANOVA  Group p = {welch_result["p-unc"].values[0]:.6f}')
print(f'  Kruskal-Wallis       p = {kw_p:.6f}')
print('All three tests converge.' if all(p < 0.05 for p in [
    anova_table.loc['C(Group)', 'PR(>F)'],
    welch_result['p-unc'].values[0],
    kw_p
]) else 'Tests do NOT all agree -- interpret with caution.')

### 3.4  H2 -- Moderated Regression: Learning Gain ~ Group x WMC_Composite

Testing whether **WMC_Composite moderates** the effect of Group on Learning Gain.

In [ ]:
# -- 3.4.1  Prepare variables ---
df['Group_coded'] = df['Group'].map({'Experimental': 1, 'Control': 0})

# Center WMC_Composite for easier interpretation of interaction
df['WMC_c'] = df['WMC_Composite'] - df['WMC_Composite'].mean()

# Interaction term
df['Group_x_WMC'] = df['Group_coded'] * df['WMC_c']

In [ ]:
# -- 3.4.2  Moderated regression ---
X = df[['Group_coded', 'WMC_c', 'Group_x_WMC']]
X = sm.add_constant(X)
y = df['Learning_Gain']

mod_reg = sm.OLS(y, X).fit()
print(mod_reg.summary())

In [ ]:
# -- 3.4.3  Interpret interaction coefficient ---
interaction_coef = mod_reg.params['Group_x_WMC']
interaction_p_reg = mod_reg.pvalues['Group_x_WMC']

print(f'Interaction coefficient (Group x WMC_Composite): {interaction_coef:.4f}')
print(f'p-value: {interaction_p_reg:.4f}')

if interaction_p_reg < 0.05:
    print('\nSIGNIFICANT moderation: The effect of Group on Learning Gain')
    print('depends on WMC level.')
    if interaction_coef > 0:
        print('Higher-WMC students benefit MORE from the Experimental intervention.')
    else:
        print('Lower-WMC students benefit MORE from the Experimental intervention.')
else:
    print('\nNo significant moderation detected.')

In [ ]:
# -- 3.5.1  Regression Diagnostics: VIF (Multicollinearity) ---
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data['Variable'] = ['Group_coded', 'WMC_c', 'Group_x_WMC']
vif_data['VIF'] = [
    variance_inflation_factor(X[['Group_coded', 'WMC_c', 'Group_x_WMC']].values, i)
    for i in range(3)
]
print('Variance Inflation Factors (VIF)')
print(vif_data.to_string(index=False))
print('\nVIF < 5: no multicollinearity concern.' if all(vif_data['VIF'] < 5)
      else 'VIF >= 5 detected: potential multicollinearity.')

In [ ]:
# -- 3.5.2  Regression Diagnostics: Residual Plots ---
residuals = mod_reg.resid
fitted = mod_reg.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Residuals vs Fitted
axes[0].scatter(fitted, residuals, alpha=0.4, s=20, color='steelblue')
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

# Q-Q Plot
from scipy.stats import probplot
probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot of Residuals')

# Histogram of Residuals
axes[2].hist(residuals, bins=25, edgecolor='black', color='steelblue', alpha=0.7)
axes[2].set_xlabel('Residuals')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Distribution of Residuals')

plt.tight_layout()
plt.savefig('residual_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: residual_diagnostics.png')

# Durbin-Watson test for autocorrelation
from statsmodels.stats.stattools import durbin_watson
dw = durbin_watson(residuals)
print(f'Durbin-Watson statistic: {dw:.4f} (close to 2 = no autocorrelation)')

In [ ]:
# -- 3.4.4  Simple slopes visualisation ---
fig, ax = plt.subplots(figsize=(8, 5))

for grp, color, marker in [('Experimental', 'steelblue', 'o'), ('Control', 'coral', 's')]:
    subset = df[df['Group'] == grp]
    ax.scatter(subset['WMC_Composite'], subset['Learning_Gain'],
               alpha=0.4, label=grp, color=color, marker=marker, s=30)
    # Regression line
    z = np.polyfit(subset['WMC_Composite'], subset['Learning_Gain'], 1)
    p_line = np.poly1d(z)
    x_range = np.linspace(subset['WMC_Composite'].min(), subset['WMC_Composite'].max(), 100)
    ax.plot(x_range, p_line(x_range), color=color, linewidth=2)

ax.set_xlabel('WMC Composite (z-score)')
ax.set_ylabel('Learning Gain')
ax.set_title('Simple Slopes: Learning Gain vs WMC by Group')
ax.legend()
plt.tight_layout()
plt.savefig('simple_slopes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: simple_slopes.png')

---
## Phase 4 — Machine Learning Classification

**Target:** `Learning_Gain_Binary` (above vs below median gain)  
**Features:** Group, WMC_Composite, WMC_Level, Pre_Content_Score, age  
**Models:** Random Forest, SVM, XGBoost  
**Evaluation:** 10-Fold Stratified Cross-Validation

In [ ]:
# -- 4.1  Prepare ML dataset ---
median_gain = df['Learning_Gain'].median()
df['Learning_Gain_Binary'] = (df['Learning_Gain'] >= median_gain).astype(int)

print(f'Median Learning Gain: {median_gain:.2f}')
print('Target distribution:')
print(df['Learning_Gain_Binary'].value_counts().rename({0: 'Below median', 1: 'At/above median'}))

# Encode categorical features
le_group = LabelEncoder()
le_wmc   = LabelEncoder()
le_gender = LabelEncoder()

df['Group_enc']     = le_group.fit_transform(df['Group'])
df['WMC_Level_enc'] = le_wmc.fit_transform(df['WMC_Level'].astype(str))
df['Gender_enc']    = le_gender.fit_transform(df['gender_id'])

FEATURES = ['Group_enc', 'WMC_Composite', 'WMC_Level_enc',
            'Pre_Content_Score', 'age_1', 'Gender_enc']
TARGET = 'Learning_Gain_Binary'

X_ml = df[FEATURES].values
y_ml = df[TARGET].values

# Scale features for SVM
scaler = StandardScaler()
X_ml_scaled = scaler.fit_transform(X_ml)

print(f'\nFeature matrix shape: {X_ml.shape}')
print(f'Features used: {FEATURES}')

In [ ]:
# -- 4.2  Define models & cross-validation ---
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
    ),
    'SVM (RBF)': SVC(
        kernel='rbf', C=1.0, gamma='scale', random_state=42
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=42, eval_metric='logloss', verbosity=0
    )
}

print('Models configured:')
for name in models:
    print(f'  {name}')

In [ ]:
# -- 4.3  Run 10-Fold CV & collect results ---
results = {}

for name, model in models.items():
    # Use scaled features for SVM, raw for tree-based
    X_use = X_ml_scaled if 'SVM' in name else X_ml

    # Cross-validated accuracy scores
    scores = cross_val_score(model, X_use, y_ml, cv=cv, scoring='accuracy')

    # Cross-validated predictions (for confusion matrix)
    y_pred = cross_val_predict(model, X_use, y_ml, cv=cv)

    results[name] = {
        'scores': scores,
        'mean_acc': scores.mean(),
        'std_acc': scores.std(),
        'y_pred': y_pred
    }

    print(f'{name:>15}:  Accuracy = {scores.mean():.4f} +/- {scores.std():.4f}  '
          f'(folds: {np.round(scores, 3)})')

In [ ]:
# -- 4.4  Classification reports ---
for name, res in results.items():
    print(f'Classification Report -- {name}')
    print(classification_report(y_ml, res['y_pred'],
                                target_names=['Below Median', 'At/Above Median']))
    print()

In [ ]:
# -- 4.5  Confusion matrices ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_ml, res['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Below', 'At/Above'])
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    ax.set_title(f'{name}\nAcc={res["mean_acc"]:.3f}+/-{res["std_acc"]:.3f}')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: confusion_matrices.png')

In [ ]:
# -- 4.6  Feature importance (Random Forest & XGBoost) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, name in zip(axes, ['Random Forest', 'XGBoost']):
    model = models[name]
    model.fit(X_ml, y_ml)
    importances = model.feature_importances_
    idx = np.argsort(importances)[::-1]

    ax.barh([FEATURES[i] for i in idx], importances[idx], color='steelblue')
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'{name} -- Feature Importance')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: feature_importance.png')

In [ ]:
# -- 4.7  Model comparison bar chart ---
fig, ax = plt.subplots(figsize=(8, 5))

model_names = list(results.keys())
means  = [results[n]['mean_acc'] for n in model_names]
stds   = [results[n]['std_acc']  for n in model_names]

bars = ax.bar(model_names, means, yerr=stds, capsize=8,
              color=['steelblue', 'coral', 'seagreen'], edgecolor='black')

for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.005,
            f'{m:.3f}', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Accuracy (10-Fold CV)')
ax.set_title('Model Comparison -- 10-Fold Cross-Validation Accuracy')
ax.set_ylim(0, 1.05)
ax.axhline(y=0.5, color='grey', linestyle='--', alpha=0.5, label='Chance level')
ax.legend()
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: model_comparison.png')

---
## Summary & Export

In [ ]:
# -- Export cleaned & engineered dataset ---
export_cols = [
    '# ms_ID', 'grade_name', 'class_name',
    'nationality_name', 'gender_id', 'Group', 'age_1',
    'Corsi_Score', 'DigitSpan_Score', 'Ospan_Score',
    'z_Corsi', 'z_DigitSpan', 'z_Ospan',
    'WMC_Composite', 'WMC_Level',
    'Pre_Content_Score', 'Post_Content_Score', 'Learning_Gain',
    'Learning_Gain_Binary'
]

df[export_cols].to_csv('data/cleaned_merged_dataset.csv', index=False, encoding='utf-8-sig')
print(f'Cleaned dataset exported: data/cleaned_merged_dataset.csv  ({len(df)} rows)')

In [ ]:
# -- Final Summary ---
print('ANALYSIS SUMMARY')
print(f'Total students (after QC):  {len(df)}')
print(f'Experimental group:         {(df["Group"] == "Experimental").sum()}')
print(f'Control group:              {(df["Group"] == "Control").sum()}')
print(f'Ospan math <85% excluded:   {n_excluded_qc} (from QC step)')
print()
print('H1: Two-Way ANOVA (Group x WMC_Level -> Learning Gain)')
for effect in ['C(Group)', 'C(WMC_Level)', 'C(Group):C(WMC_Level)']:
    F_val = anova_table.loc[effect, 'F']
    p_val = anova_table.loc[effect, 'PR(>F)']
    eta   = anova_table.loc[effect, 'partial_eta_sq']
    sig   = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    print(f'  {effect:<30} F={F_val:>7.3f}  p={p_val:.4f} {sig}  eta_p={eta:.4f}')
print()
print('H2: Moderated Regression')
print(f'  Interaction (Group x WMC):  B={interaction_coef:.4f}, p={interaction_p_reg:.4f}')
print(f'  R2 = {mod_reg.rsquared:.4f}, Adj. R2 = {mod_reg.rsquared_adj:.4f}')
print()
print('ML Classification (10-Fold CV Accuracy)')
for name, res in results.items():
    print(f'  {name:<15} {res["mean_acc"]:.4f} +/- {res["std_acc"]:.4f}')
print()
best_model = max(results, key=lambda k: results[k]['mean_acc'])
print(f'Best model: {best_model} ({results[best_model]["mean_acc"]:.4f})')

In [ ]:
# -- Export all results to CSV tables ---
import os
OUT_DIR = 'results/'
os.makedirs(OUT_DIR, exist_ok=True)

# 1. Cleaned merged dataset
df[export_cols].to_csv(OUT_DIR + 'cleaned_merged_dataset.csv', index=False, encoding='utf-8-sig')

# 2. Descriptive statistics by Group
desc_all = df[numeric_cols].describe().round(3)
desc_exp = df[df['Group'] == 'Experimental'][numeric_cols].describe().round(3)
desc_ctrl = df[df['Group'] == 'Control'][numeric_cols].describe().round(3)
desc_all.to_csv(OUT_DIR + 'descriptive_statistics_all.csv', encoding='utf-8-sig')
desc_exp.to_csv(OUT_DIR + 'descriptive_statistics_experimental.csv', encoding='utf-8-sig')
desc_ctrl.to_csv(OUT_DIR + 'descriptive_statistics_control.csv', encoding='utf-8-sig')

# 3. Group and demographic counts
group_counts = df['Group'].value_counts().reset_index()
group_counts.columns = ['Group', 'Count']
group_counts.to_csv(OUT_DIR + 'group_counts.csv', index=False, encoding='utf-8-sig')

gender_counts = df['gender_id'].value_counts().reset_index()
gender_counts.columns = ['Gender', 'Count']
gender_counts.to_csv(OUT_DIR + 'gender_counts.csv', index=False, encoding='utf-8-sig')

grade_counts = df['grade_name'].value_counts().reset_index()
grade_counts.columns = ['Grade', 'Count']
grade_counts.to_csv(OUT_DIR + 'grade_counts.csv', index=False, encoding='utf-8-sig')

# 4. WMC Level x Group crosstab
wmc_group_ct = pd.crosstab(df['WMC_Level'], df['Group'])
wmc_group_ct.to_csv(OUT_DIR + 'wmc_level_group_crosstab.csv', encoding='utf-8-sig')

# 5. Correlation matrix
corr_matrix.to_csv(OUT_DIR + 'correlation_matrix.csv', encoding='utf-8-sig')

# 6. Outlier detection results
outlier_df.to_csv(OUT_DIR + 'outlier_detection_iqr.csv', index=False, encoding='utf-8-sig')

# 7. Normality test results (Shapiro-Wilk)
normality_df.to_csv(OUT_DIR + 'shapiro_wilk_normality.csv', index=False, encoding='utf-8-sig')

# 8. Levene's test
levene_df = pd.DataFrame([{
    'Test': "Levene's",
    'Statistic': round(lev_stat, 4),
    'p-value': round(lev_p, 6),
    'Homogeneity assumed': 'Yes' if lev_p > 0.05 else 'No'
}])
levene_df.to_csv(OUT_DIR + 'levene_test.csv', index=False, encoding='utf-8-sig')

# 9. Two-Way ANOVA table with effect sizes
anova_export = anova_table[['sum_sq', 'df', 'F', 'PR(>F)', 'eta_sq', 'partial_eta_sq']].copy()
anova_export = anova_export.round(6)
anova_export.to_csv(OUT_DIR + 'anova_two_way.csv', encoding='utf-8-sig')

# 10. Robustness checks (Welch's ANOVA + Kruskal-Wallis + Mann-Whitney)
robustness_df = pd.DataFrame([
    {'Test': 'Standard ANOVA (Group)', 'Statistic': round(anova_table.loc['C(Group)', 'F'], 4),
     'p-value': round(anova_table.loc['C(Group)', 'PR(>F)'], 6),
     'Effect size': round(anova_table.loc['C(Group)', 'partial_eta_sq'], 4),
     'Effect size type': 'partial eta-sq'},
    {'Test': "Welch's ANOVA (Group)", 'Statistic': round(welch_result['F'].values[0], 4),
     'p-value': round(welch_result['p-unc'].values[0], 6),
     'Effect size': round(welch_result['np2'].values[0], 4),
     'Effect size type': 'partial eta-sq'},
    {'Test': 'Kruskal-Wallis (Group)', 'Statistic': round(kw_stat, 4),
     'p-value': round(kw_p, 6),
     'Effect size': None,
     'Effect size type': None},
    {'Test': 'Mann-Whitney U (Group)', 'Statistic': round(u_stat, 1),
     'p-value': round(u_p, 6),
     'Effect size': round(rank_biserial, 4),
     'Effect size type': 'rank-biserial r'}
])
robustness_df.to_csv(OUT_DIR + 'robustness_checks.csv', index=False, encoding='utf-8-sig')

# 11. Moderated regression coefficients
reg_coefs = pd.DataFrame({
    'Variable': mod_reg.params.index,
    'Coefficient': mod_reg.params.round(4).values,
    'Std Error': mod_reg.bse.round(4).values,
    't-value': mod_reg.tvalues.round(4).values,
    'p-value': mod_reg.pvalues.round(6).values,
    'CI lower': mod_reg.conf_int()[0].round(4).values,
    'CI upper': mod_reg.conf_int()[1].round(4).values
})
reg_coefs.to_csv(OUT_DIR + 'moderated_regression_coefficients.csv', index=False, encoding='utf-8-sig')

# 12. Regression model summary
reg_summary = pd.DataFrame([{
    'R-squared': round(mod_reg.rsquared, 4),
    'Adj R-squared': round(mod_reg.rsquared_adj, 4),
    'F-statistic': round(mod_reg.fvalue, 4),
    'F p-value': round(mod_reg.f_pvalue, 6),
    'AIC': round(mod_reg.aic, 2),
    'BIC': round(mod_reg.bic, 2),
    'Durbin-Watson': round(dw, 4),
    'N': int(mod_reg.nobs)
}])
reg_summary.to_csv(OUT_DIR + 'moderated_regression_summary.csv', index=False, encoding='utf-8-sig')

# 13. VIF results
vif_data.to_csv(OUT_DIR + 'vif_multicollinearity.csv', index=False, encoding='utf-8-sig')

# 14. ML cross-validation results
ml_results = pd.DataFrame([{
    'Model': name,
    'Mean Accuracy': round(res['mean_acc'], 4),
    'Std Accuracy': round(res['std_acc'], 4),
    **{f'Fold_{i+1}': round(res['scores'][i], 4) for i in range(10)}
} for name, res in results.items()])
ml_results.to_csv(OUT_DIR + 'ml_cv_results.csv', index=False, encoding='utf-8-sig')

# 15. Classification reports per model
for name, res in results.items():
    cr = classification_report(y_ml, res['y_pred'],
                               target_names=['Below Median', 'At/Above Median'],
                               output_dict=True)
    cr_df = pd.DataFrame(cr).T.round(4)
    safe_name = name.replace(' ', '_').replace('(', '').replace(')', '')
    cr_df.to_csv(OUT_DIR + f'classification_report_{safe_name}.csv', encoding='utf-8-sig')

# 16. Confusion matrices
for name, res in results.items():
    cm = confusion_matrix(y_ml, res['y_pred'])
    cm_df = pd.DataFrame(cm, index=['Actual: Below', 'Actual: At/Above'],
                             columns=['Predicted: Below', 'Predicted: At/Above'])
    safe_name = name.replace(' ', '_').replace('(', '').replace(')', '')
    cm_df.to_csv(OUT_DIR + f'confusion_matrix_{safe_name}.csv', encoding='utf-8-sig')

# 17. Feature importance (Random Forest & XGBoost)
for name in ['Random Forest', 'XGBoost']:
    model = models[name]
    imp_df = pd.DataFrame({
        'Feature': FEATURES,
        'Importance': model.feature_importances_.round(4)
    }).sort_values('Importance', ascending=False)
    safe_name = name.replace(' ', '_')
    imp_df.to_csv(OUT_DIR + f'feature_importance_{safe_name}.csv', index=False, encoding='utf-8-sig')

# 18. Learning Gain by Group x WMC Level
gain_by_group_wmc = df.groupby(['Group', 'WMC_Level'])['Learning_Gain'].agg(
    ['count', 'mean', 'std', 'min', 'median', 'max']
).round(3)
gain_by_group_wmc.to_csv(OUT_DIR + 'learning_gain_by_group_wmc.csv', encoding='utf-8-sig')

# List all exported files
exported = sorted(os.listdir(OUT_DIR))
print(f'Exported {len(exported)} CSV files to {OUT_DIR}')
for f in exported:
    size_kb = os.path.getsize(OUT_DIR + f) / 1024
    print(f'  {f} ({size_kb:.1f} KB)')